In [21]:
import os
from os.path import dirname
from parser import *
from preprocessing import *
from indexing import *
from search import *
from weighting_methods import *
from utils import *
from doc2vec_rank import *
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer
from nltk.stem.snowball import EnglishStemmer

In [ ]:
# Two setups with the most relevant documents returned are
# 1. tfidf_raw_score_ink_wrds_lancaster   308/339
# 2. tfidf_raw_score_nltk_wrds_lancaster  307/339

# booleans to control parsing
parse_docs = False
parse_queries = False

#dataset logistics
absolute_base_path = "C:\\Users\\awyat049\\OneDrive\\Ottawa U\\Year 5\\2025 Winter\\MAT4107 - Information Retrieval and Internet\\Assignments\\A2\\KHES_BRANCH\\InfoRetrievNeural"
dataset = absolute_base_path + "\\data\\scifact" #this is where we will change the dataset that we use
doc_file_path = dataset + '\\corpus.jsonl'
query_file_path = dataset + '\\queries.jsonl'
results_file_path = absolute_base_path + "\\eval\\trec_eval-9.0.7\\test"

# Processed files
index_file_path = absolute_base_path + '\\data\\processed\\inverted_index.json'


# Define which stopwords list to use
# load in stopword files - 179 words
#nltk.download('stopwords')
#nltk.download('punkt_tab')
#using a set as it is easier to look up things from (in O(1) as opposed to O(n) from a list)
#stop_words1 = set(stopwords.words('english'))

# read in StopWords List - 779 words
stop_words2 = set()
with open("C:\\Users\\awyat049\\OneDrive\\Ottawa U\\Year 5\\2025 Winter\\MAT4107 - Information Retrieval and Internet\\Assignments\\A2\\KHES_BRANCH\\InfoRetrievNeural\\data" + "\\StopWords.txt") as file:
    for line in file:
        stop_words2.add(line.rstrip())

# define stopwords
stop_words = [stop_words2]
stop_words_labels = ["ink_wrds"]


# Define stemmers
stemmers = [LancasterStemmer()]
stemmer_labels = ["lancaster"]

parsed_docs = []
parsed_queries = []
descriptors = []

# preprocess documents and queries for all possible combos of stop words selection and stemmers
for stop_wordi in range(len(stop_words)):
    for stemmeri in range(len(stemmers)):
        descriptors.append(stop_words_labels[stop_wordi] + '_' + stemmer_labels[stemmeri])

        preprocessed_docs_path = absolute_base_path + '\\data\\processed\\preprocessed_docs_' + stop_words_labels[stop_wordi] + '_' + stemmer_labels[stemmeri] + '.json'
        preprocessed_queries_path = absolute_base_path + '\\data\\processed\\preprocessed_queries_' + stop_words_labels[stop_wordi] + '_' + stemmer_labels[stemmeri] + '.json'
        print(f"Parsing the dataset with stopwords = {stop_words_labels[stop_wordi]} and stemmer = {stemmer_labels[stemmeri]}...")
        documents=[]
        queries = []

        

        #preprocessing the documents
        if os.path.exists(preprocessed_docs_path) and not parse_docs:
            print("Loading preprocessed documents...")
            documents = load_preprocessed_data(preprocessed_docs_path)
        else:
            print("Preprocessing documents...")
            # change params here to use different stemmer and different stop words list / to not use either
            documents = preprocess_documents(parse_documents_from_file(doc_file_path), removestopwords=True, stopwords=stop_words[stop_wordi], stem_text=True, stemmer = stemmers[stemmeri])
            save_preprocessed_data(documents, preprocessed_docs_path)
        
        parsed_docs.append(documents)

        #Preprocessing the queries if they have not been preprocessed yet
        if os.path.exists(preprocessed_queries_path) and not parse_queries:
            print("Loading preprocessed queries...")
            queries=load_preprocessed_data(preprocessed_queries_path)
        else:
            print("Preprocessing queries...")
            queries = preprocess_queries(parse_queries_from_file(query_file_path), removestopwords=True, stopwords=stop_words[stop_wordi], stem_text=True, stemmer = stemmers[stemmeri])
            save_preprocessed_data(queries, preprocessed_queries_path)
        
        parsed_queries.append(queries)

print("Done Preprocessing")

# define similarity measures
sim_measures = ["raw_score"]

inverted_indices = []

# loop through all preprocessed documents and create an inverted index for each
for doc in parsed_docs:
    # build inverted index
    inverted_indices.append(invert_index(doc))
print("Done Inverted Indices")

outputs = []

count = 0
for invi in range(len(inverted_indices)):
    # define weight methods
    weight_mthds = [tf_idf(inverted_indices[invi], doc_lengths=collect_doc_lengths(parsed_docs[invi]))]
    weight_mthds_lbls = ["tfidf"]

    for mthdi in range(len(weight_mthds)):
        for sim_measure in sim_measures:
            count += 1

            search_e = SearchEngine(weight_mthds[mthdi], similarity_measure = sim_measure)
            search_e.search(pair_usable_query(parsed_queries[invi]))
            print(f"Done Search {count}")

            #convert_output_form(search_e.results, "test1").to_csv(results_file_path + "\\test_out.txt", header = None, index = None, sep = ' ')
            output = convert_output_form(search_e.results, weight_mthds_lbls[mthdi] + '_' + sim_measure + '_' + descriptors[invi])

            outputs.append(output)

            save_list_output(output, results_file_path + "\\" + weight_mthds_lbls[mthdi] + '_' + sim_measure + '_' + descriptors[invi] + ".test")

#save_inv_index(inverted_index,path) #replace path for the path you want to save inverted index to

#print(search_e.results)

# create index of corpus and queries without preprocessing (for nn models)
parsed_do = parse_documents_from_file_nn(doc_file_path)
parsed_quer = parse_queries_from_file_nn(query_file_path)

#print(parsed_quer.keys())

vec_size = 50 # change to 300 for tests
epochs = 5 # change to 50 for tests
docvec = Doc2Vector(vector_size=vec_size, epochs = epochs)
docvec.train_doc2vec(parsed_do)

Parsing the dataset with stopwords = ink_wrds and stemmer = lancaster...
Loading preprocessed documents...
Loading preprocessed queries...
Done Preprocessing
Done Inverted Indices
Done Search 1
{'0': '0-dimensional biomaterials lack inductive properties.', '2': '1 in 5 million in UK have abnormal PrP positivity.', '4': '1-1% of colorectal cancer patients are diagnosed with regional or distant metastases.', '6': '10% of sudden infant death syndrome (SIDS) deaths happen in newborns aged less than 6 months.', '9': '32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.', '10': '4-PBA treatment decreases endoplasmic reticulum stress in response to general endoplasmic reticulum stress markers.', '11': '4-PBA treatment raises endoplasmic reticulum stress in response to general endoplasmic reticulum stress markers.', '12': '40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.', '1

In [ ]:
docvec.model.docvecs[19238]

C:\Users\awyat049\AppData\Local\Temp\ipykernel_27504\536935745.py:1: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  docvec.model.docvecs['19238']


KeyError: "Key '19238' not present"

In [12]:
docvec.model.corpus_count

5183

---
# BERT Testing

In [2]:
import torch

In [3]:
torch.cuda.is_available()

True

In [6]:
test = None
if test is None:
    print("yes")

yes


In [9]:
test = "hello"
if test is not None:
    print("yes")


yes


In [15]:
tes = torch.tensor([[0.0019]])

In [20]:
tes.item()

0.0019000000320374966

In [33]:
# Two setups with the most relevant documents returned are
# 1. tfidf_raw_score_ink_wrds_lancaster   308/339
# 2. tfidf_raw_score_nltk_wrds_lancaster  307/339

# booleans to control parsing
parse_docs = False
parse_queries = False

#dataset logistics
absolute_base_path = "C:\\Users\\awyat049\\OneDrive\\Ottawa U\\Year 5\\2025 Winter\\MAT4107 - Information Retrieval and Internet\\Assignments\\A2\\KHES_BRANCH\\InfoRetrievNeural"
dataset = absolute_base_path + "\\data\\scifact" #this is where we will change the dataset that we use
doc_file_path = dataset + '\\corpus.jsonl'
query_file_path = dataset + '\\queries.jsonl'
results_file_path = absolute_base_path + "\\eval\\trec_eval-9.0.7\\test"

# Processed files
index_file_path = absolute_base_path + '\\data\\processed\\inverted_index.json'


# Define which stopwords list to use
# load in stopword files - 179 words
#nltk.download('stopwords')
#nltk.download('punkt_tab')
#using a set as it is easier to look up things from (in O(1) as opposed to O(n) from a list)
#stop_words1 = set(stopwords.words('english'))

# read in StopWords List - 779 words
stop_words2 = set()
with open("C:\\Users\\awyat049\\OneDrive\\Ottawa U\\Year 5\\2025 Winter\\MAT4107 - Information Retrieval and Internet\\Assignments\\A2\\KHES_BRANCH\\InfoRetrievNeural\\data" + "\\StopWords.txt") as file:
    for line in file:
        stop_words2.add(line.rstrip())

# define stopwords
stop_words = [stop_words2]
stop_words_labels = ["ink_wrds"]


# Define stemmers
stemmers = [LancasterStemmer()]
stemmer_labels = ["lancaster"]

parsed_docs = []
parsed_queries = []
descriptors = []

# preprocess documents and queries for all possible combos of stop words selection and stemmers
for stop_wordi in range(len(stop_words)):
    for stemmeri in range(len(stemmers)):
        descriptors.append(stop_words_labels[stop_wordi] + '_' + stemmer_labels[stemmeri])

        preprocessed_docs_path = absolute_base_path + '\\data\\processed\\preprocessed_docs_' + stop_words_labels[stop_wordi] + '_' + stemmer_labels[stemmeri] + '.json'
        preprocessed_queries_path = absolute_base_path + '\\data\\processed\\preprocessed_queries_' + stop_words_labels[stop_wordi] + '_' + stemmer_labels[stemmeri] + '.json'
        print(f"Parsing the dataset with stopwords = {stop_words_labels[stop_wordi]} and stemmer = {stemmer_labels[stemmeri]}...")
        documents=[]
        queries = []

        

        #preprocessing the documents
        if os.path.exists(preprocessed_docs_path) and not parse_docs:
            print("Loading preprocessed documents...")
            documents = load_preprocessed_data(preprocessed_docs_path)
        else:
            print("Preprocessing documents...")
            # change params here to use different stemmer and different stop words list / to not use either
            documents = preprocess_documents(parse_documents_from_file(doc_file_path), removestopwords=True, stopwords=stop_words[stop_wordi], stem_text=True, stemmer = stemmers[stemmeri])
            save_preprocessed_data(documents, preprocessed_docs_path)
        
        parsed_docs.append(documents)

        #Preprocessing the queries if they have not been preprocessed yet
        if os.path.exists(preprocessed_queries_path) and not parse_queries:
            print("Loading preprocessed queries...")
            queries=load_preprocessed_data(preprocessed_queries_path)
        else:
            print("Preprocessing queries...")
            queries = preprocess_queries(parse_queries_from_file(query_file_path), removestopwords=True, stopwords=stop_words[stop_wordi], stem_text=True, stemmer = stemmers[stemmeri])
            save_preprocessed_data(queries, preprocessed_queries_path)
        
        parsed_queries.append(queries)

print("Done Preprocessing")

# define similarity measures
sim_measures = ["raw_score"]

inverted_indices = []

# loop through all preprocessed documents and create an inverted index for each
for doc in parsed_docs:
    # build inverted index
    inverted_indices.append(invert_index(doc))
print("Done Inverted Indices")

outputs = []

count = 0
for invi in range(len(inverted_indices)):
    # define weight methods
    weight_mthds = [tf_idf(inverted_indices[invi], doc_lengths=collect_doc_lengths(parsed_docs[invi]))]
    weight_mthds_lbls = ["tfidf"]

    for mthdi in range(len(weight_mthds)):
        for sim_measure in sim_measures:
            count += 1

            search_e = SearchEngine(weight_mthds[mthdi], similarity_measure = sim_measure)
            search_e.search(pair_usable_query(parsed_queries[invi]))
            print(f"Done Search {count}")

            #convert_output_form(search_e.results, "test1").to_csv(results_file_path + "\\test_out.txt", header = None, index = None, sep = ' ')
            output = convert_output_form(search_e.results, weight_mthds_lbls[mthdi] + '_' + sim_measure + '_' + descriptors[invi])

            outputs.append(output)

            save_list_output(output, results_file_path + "\\" + weight_mthds_lbls[mthdi] + '_' + sim_measure + '_' + descriptors[invi] + ".test")

#save_inv_index(inverted_index,path) #replace path for the path you want to save inverted index to

#print(search_e.results)
parsed_docs1 = parsed_docs[0]
parsed_queries1 = parsed_queries[0]

# create index of corpus and queries - removed stop words etc for doc2vec
parsed_quer = dict()
for dic in parsed_queries1:
    tmp = {dic['num']: " ".join(dic['query'])}
    parsed_quer.update(tmp)

parsed_do = dict()
for dic in parsed_docs1:
    tmp = {dic['DOCNO']: (" ".join(dic['HEAD']) + " ".join(dic['TEXT']))}
    parsed_do.update(tmp)


Parsing the dataset with stopwords = ink_wrds and stemmer = lancaster...
Loading preprocessed documents...
Loading preprocessed queries...
Done Preprocessing
Done Inverted Indices
Done Search 1


In [38]:
parsed_quer

{'0': 'dimend biom lack induc property',
 '2': 'mil uk hav abnorm prp posit',
 '4': 'colorect cant paty ar diagnos reg dist metastas',
 '6': 'sud inf dea syndrom sid death hap newborn ag month',
 '9': 'liv transpl program requir paty discontinu methadon tre',
 '10': 'pba tre decreas endoplasm reticul stress respons gen endoplasm reticul stress mark',
 '11': 'pba tre rais endoplasm reticul stress respons gen endoplasm reticul stress mark',
 '12': 'mgday fol acid mgday vitamin doe affect chronic kidney diseas ckd progress',
 '14': 'nucleotidas metabol mp',
 '15': 'paty expos rady hav act mark mesenchym stem cel',
 '17': 'perin mort bir weight',
 '18': 'colorect cant paty ar diagnos reg dist metastas',
 '19': 'dayold adult caenorhabdit eleg exhibit approxim learn capac adult',
 '20': 'dayold adult caenorhabdit eleg exhibit approxim learn capac adult',
 '21': 'burn paty ar admit hospit furth tre aft appear hospit emerg ward outpaty clin',
 '22': 'peopl sev ment disord receiv tre middl inco